# MerRec — Prepare data for Recommendation Models

Notebook này tạo một bộ dữ liệu sạch dùng chung cho các model, chia train/validation/test theo thời gian, tạo Popularity/Trending, item catalog, implicit-feedback cho CF và các cohort cold-start.

## 0. Cấu hình

In [1]:
from pathlib import Path
from collections import Counter
import os
import shutil
import math

import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# ============================================================
# DATA PATH
# ============================================================
LOCAL_DATA_PATH = Path(r"D:\MerRec\data\raw\20230501")

if LOCAL_DATA_PATH.exists():
    DATA_PATH = LOCAL_DATA_PATH
elif Path("/kaggle/input").exists():
    parquet_files = list(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("Không tìm thấy parquet trong /kaggle/input")
    parents = Counter(p.parent for p in parquet_files)
    DATA_PATH = max(parents, key=parents.get)
else:
    raise FileNotFoundError(
        "Không tìm thấy dataset. Sửa LOCAL_DATA_PATH hoặc add dataset vào Kaggle."
    )

# Output
if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/merrec_processed")
else:
    OUT_DIR = DATA_PATH.parents[1] / "processed" / "recommender"

TEMP_DIR = OUT_DIR / "duckdb_temp"
TABLE_DIR = OUT_DIR / "tables"
MODEL_DIR = OUT_DIR / "model_data"
SERVING_DIR = OUT_DIR / "serving"

# Output paths are declared once here so every later cell can reuse them.
INTERACTIONS_DIR = MODEL_DIR / "interactions"

user_map_path = MODEL_DIR / "train_user_map.parquet"
item_map_path = MODEL_DIR / "train_item_map.parquet"

catalog_train_path = MODEL_DIR / "item_catalog_train.parquet"
catalog_full_path = SERVING_DIR / "item_catalog_full.parquet"

cf_train_path = MODEL_DIR / "cf_train.parquet"
pop_train_path = MODEL_DIR / "popularity_train.parquet"
trend_train_path = MODEL_DIR / "trending_train.parquet"

pop_serving_path = SERVING_DIR / "popularity_snapshot.parquet"
trend_serving_path = SERVING_DIR / "trending_snapshot.parquet"
home_feed_path = SERVING_DIR / "home_popular_trending.parquet"
category_trending_path = SERVING_DIR / "trending_by_category.parquet"

eval_cohort_path = MODEL_DIR / "eval_events_with_cold_start.parquet"

for p in [OUT_DIR, TEMP_DIR, TABLE_DIR, MODEL_DIR, SERVING_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# SPLIT
# ============================================================
# Với dữ liệu 1 tháng: 3 ngày cuối = test, 3 ngày trước = validation.
VAL_DAYS = 3
TEST_DAYS = 3

# Trending time decay
TREND_HALF_LIFE_DAYS = 3.0

# Dedupe cùng user-item-event-timestamp-session-sequence
DROP_EXACT_EVENT_DUPLICATES = True

# Event weights: dùng cho implicit CF / popularity / trending.
EVENT_WEIGHTS = {
    "view": 1.0,
    "like": 2.0,
    "cart": 4.0,
    "offer": 5.0,
    "buy_start": 7.0,
    "buy_comp": 10.0,
}

print("DATA_PATH:", DATA_PATH)
print("OUT_DIR :", OUT_DIR)

DATA_PATH: D:\MerRec\data\raw\20230501
OUT_DIR : D:\MerRec\data\processed\recommender


## 1. DuckDB và raw view

In [2]:
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

DB_PATH = OUT_DIR / "prepare_recommender.duckdb"
con = duckdb.connect(str(DB_PATH))

threads = max(1, min(8, (os.cpu_count() or 4) - 1))
con.execute(f"SET threads={threads}")
con.execute("SET preserve_insertion_order=false")
con.execute("SET temp_directory='" + TEMP_DIR.as_posix().replace("'", "''") + "'")
con.execute("SET TimeZone='UTC'")

def q(sql):
    return con.execute(sql).df()

def show(name, df, n=20):
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{name}: {len(df):,} rows")
    display(df.head(n))
    return df

glob_path = (DATA_PATH / "*.parquet").as_posix().replace("'", "''")

con.execute(f"""
CREATE OR REPLACE VIEW raw AS
SELECT *
FROM read_parquet(
    '{glob_path}',
    union_by_name=true,
    filename=true,
    file_row_number=true
)
""")

schema = q("DESCRIBE raw")
show("00_raw_schema", schema)

required = {"user_id", "item_id", "event_id", "stime"}
missing_required = sorted(required - set(schema["column_name"]))
if missing_required:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_required}")

00_raw_schema: 27 rows


,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,stime,TIMESTAMP,YES,None,None,None
2,session_id,VARCHAR,YES,None,None,None
3,sequence_id,VARCHAR,YES,None,None,None
4,sequence_length,BIGINT,YES,None,None,None
5,event_id,VARCHAR,YES,None,None,None
6,item_id,BIGINT,YES,None,None,None
7,product_id,VARCHAR,YES,None,None,None
8,name,VARCHAR,YES,None,None,None
9,price,DOUBLE,YES,None,None,None


## 2. Chuẩn hóa event và timestamp

In [3]:
raw_columns = set(schema["column_name"])

def has(col):
    return col in raw_columns

def raw_or_null(col):
    return f'"{col}"' if has(col) else "NULL"

def txt(col):
    return f"NULLIF(TRIM(CAST({raw_or_null(col)} AS VARCHAR)), '')"

# ============================================================
# TIMESTAMP PARSER ROBUST
# ============================================================
EPOCH_UNIT = "ms"

def stime_expression(data_type, epoch_unit="ms"):
    data_type = str(data_type).upper()

    if data_type.startswith("TIMESTAMP") or data_type == "DATE":
        return "TRY_CAST(stime AS TIMESTAMP)"

    divisor = {
        "s": 1,
        "ms": 1_000,
        "us": 1_000_000,
        "ns": 1_000_000_000,
    }[epoch_unit]

    epoch_sql = (
        f"TRY(CAST("
        f"to_timestamp(TRY_CAST(stime AS DOUBLE) / {divisor}) "
        f"AS TIMESTAMP))"
    )

    if any(t in data_type for t in [
        "INT", "HUGEINT", "UBIGINT",
        "DOUBLE", "FLOAT", "DECIMAL", "REAL"
    ]):
        return epoch_sql

    # Nếu stime là VARCHAR:
    # - chuỗi số -> Unix epoch
    # - chuỗi datetime -> parser timestamp
    return f"""
    CASE
        WHEN TRY_CAST(stime AS DOUBLE) IS NOT NULL
            THEN {epoch_sql}
        ELSE TRY_CAST(TRY_CAST(stime AS TIMESTAMPTZ) AS TIMESTAMP)
    END
    """

stime_type = (
    q("DESCRIBE raw")
    .set_index("column_name")
    .loc["stime", "column_type"]
)

ts_sql = stime_expression(stime_type, EPOCH_UNIT)

print("stime dtype:", stime_type)
print("epoch unit :", EPOCH_UNIT)

timestamp_sample = q(f"""
SELECT
    stime AS original_stime,
    TRY_CAST(stime AS DOUBLE) AS stime_as_double,
    {ts_sql} AS parsed_stime
FROM raw
WHERE stime IS NOT NULL
LIMIT 20
""")

show("01_timestamp_parse_sample", timestamp_sample)

timestamp_check = q(f"""
SELECT
    count(*) AS total_rows,
    count(*) FILTER (WHERE stime IS NULL) AS source_null_stime,
    count(*) FILTER (
        WHERE stime IS NOT NULL
          AND ({ts_sql}) IS NULL
    ) AS parse_failed,
    min({ts_sql}) AS min_parsed_ts,
    max({ts_sql}) AS max_parsed_ts
FROM raw
""")

show("01_timestamp_parse_quality", timestamp_check)

if int(timestamp_check.loc[0, "parse_failed"]) > 0:
    fail_pct = (
        100.0 * int(timestamp_check.loc[0, "parse_failed"])
        / max(1, int(timestamp_check.loc[0, "total_rows"]))
    )
    print(f"⚠️ Timestamp parse failed: {fail_pct:.6f}%")

if pd.isna(timestamp_check.loc[0, "min_parsed_ts"]) or pd.isna(timestamp_check.loc[0, "max_parsed_ts"]):
    raise ValueError(
        "Không parse được stime. Dừng preprocessing để tránh chia train/val/test sai."
    )

event_norm = f"LOWER({txt('event_id')})"

# ============================================================
# MISSING METADATA
# - category: fallback từ cấp con -> cấp cha
# - brand/condition/shipper: missing -> __UNK__
# - size: giữ nguyên NULL vì missing rất cao; model nào cần sẽ xử lý riêng
# - color: giữ ở standardized để serving có thể dùng, nhưng KHÔNG xuất
#   vào train/validation/test.
# ============================================================
category0_sql = f"COALESCE({txt('c0_name')}, '__UNK__')"
category1_sql = f"COALESCE({txt('c1_name')}, {txt('c0_name')}, '__UNK__')"
category2_sql = f"COALESCE({txt('c2_name')}, {txt('c1_name')}, {txt('c0_name')}, '__UNK__')"

brand_sql = f"COALESCE({txt('brand_name')}, '__UNK__')"
condition_sql = f"COALESCE({txt('item_condition_name')}, '__UNK__')"
shipper_sql = f"COALESCE({txt('shipper_name')}, '__UNK__')"

con.execute(f"""
CREATE OR REPLACE VIEW standardized AS
SELECT
    {txt("user_id")} AS user_id,
    {txt("item_id")} AS item_id,
    {txt("session_id")} AS session_id,
    {txt("sequence_id")} AS sequence_id,
    {txt("event_id")} AS event_id,
    {ts_sql} AS ts,

CASE
    WHEN {event_norm} = 'item_view' THEN 'view'
    WHEN {event_norm} = 'item_like' THEN 'like'
    WHEN {event_norm} = 'item_add_to_cart_tap' THEN 'cart'
    WHEN {event_norm} = 'offer_make' THEN 'offer'
    WHEN {event_norm} = 'buy_start' THEN 'buy_start'
    WHEN {event_norm} = 'buy_comp' THEN 'buy_comp'
    ELSE 'other'
END AS event_group,

    CASE
        WHEN isfinite(TRY_CAST({raw_or_null("price")} AS DOUBLE))
        THEN TRY_CAST({raw_or_null("price")} AS DOUBLE)
    END AS price,

    {txt("product_id")} AS product_id,
    {txt("name")} AS name,

    {category0_sql} AS category0,
    {category1_sql} AS category1,
    {category2_sql} AS category2,

    {brand_sql} AS brand,
    {condition_sql} AS condition,

    {txt("size_name")} AS size,

    -- Chỉ giữ color trong standardized/serving; không đưa vào model train/val/test.
    {txt("color")} AS color,

    {shipper_sql} AS shipper,

    filename AS source_file,
    file_row_number AS source_row
FROM raw
""")

event_check = q("""
SELECT
    event_id,
    event_group,
    count(*) AS interactions
FROM standardized
GROUP BY 1,2
ORDER BY interactions DESC
""")

show("01_event_mapping", event_check)

if (event_check["event_group"] == "other").any():
    print("⚠️ Có event_group='other'. Xem bảng 01_event_mapping trước khi train.")


stime dtype: TIMESTAMP
epoch unit : ms
01_timestamp_parse_sample: 20 rows


,original_stime,stime_as_double,parsed_stime
0,2023-05-30 23:11:15,NaN,2023-05-30 23:11:15
1,2023-05-30 23:11:25,NaN,2023-05-30 23:11:25
2,2023-05-30 23:11:49,NaN,2023-05-30 23:11:49
3,2023-05-30 23:11:52,NaN,2023-05-30 23:11:52
4,2023-05-28 01:45:05,NaN,2023-05-28 01:45:05
5,2023-05-28 01:45:07,NaN,2023-05-28 01:45:07
6,2023-05-28 01:45:55,NaN,2023-05-28 01:45:55
7,2023-05-28 01:45:57,NaN,2023-05-28 01:45:57
8,2023-05-28 01:46:04,NaN,2023-05-28 01:46:04
9,2023-05-28 01:46:06,NaN,2023-05-28 01:46:06


01_timestamp_parse_quality: 1 rows


,total_rows,source_null_stime,parse_failed,min_parsed_ts,max_parsed_ts
0,174872167,0,0,2023-05-01,2023-05-31


01_event_mapping: 6 rows


,event_id,event_group,interactions
0,item_view,view,147856790
1,item_like,like,22271899
2,item_add_to_cart_tap,cart,2895617
3,offer_make,offer,1224425
4,buy_start,buy_start,432120
5,buy_comp,buy_comp,191316


## 3. Cleaning và xác định mốc split

In [4]:
base_quality = q("""
SELECT
    count(*) AS total_rows,
    count(*) FILTER (WHERE user_id IS NULL) AS null_user,
    count(*) FILTER (WHERE item_id IS NULL) AS null_item,
    count(*) FILTER (WHERE ts IS NULL) AS null_ts,
    min(ts) AS min_ts,
    max(ts) AS max_ts
FROM standardized
""")

show("02_base_quality", base_quality)

total_rows = int(base_quality.loc[0, "total_rows"])
null_ts = int(base_quality.loc[0, "null_ts"])

if null_ts == total_rows:
    raise ValueError(
        "100% ts đang NULL. Timestamp parser chưa đúng; không được tiếp tục split."
    )

if null_ts > 0:
    print(
        f"⚠️ Có {null_ts:,}/{total_rows:,} rows bị NULL timestamp "
        f"({100.0*null_ts/total_rows:.6f}%)."
    )

max_ts = pd.Timestamp(base_quality.loc[0, "max_ts"])
min_ts = pd.Timestamp(base_quality.loc[0, "min_ts"])

if pd.isna(max_ts) or pd.isna(min_ts):
    raise ValueError(
        "min_ts/max_ts là NaT. Dừng preprocessing trước khi chia train/val/test."
    )

max_day = max_ts.normalize()
test_start = max_day - pd.Timedelta(days=TEST_DAYS - 1)
val_start = test_start - pd.Timedelta(days=VAL_DAYS)

print("Range     :", min_ts, "→", max_ts)
print("Train     : <", val_start)
print("Validation:", val_start, "→", test_start)
print("Test      : >=", test_start)

02_base_quality: 1 rows


,total_rows,null_user,null_item,null_ts,min_ts,max_ts
0,174872167,0,0,0,2023-05-01,2023-05-31


Range     : 2023-05-01 00:00:00 → 2023-05-31 00:00:00
Train     : < 2023-05-26 00:00:00
Validation: 2023-05-26 00:00:00 → 2023-05-29 00:00:00
Test      : >= 2023-05-29 00:00:00


In [5]:
weight_case = "CASE event_group\n" + "\n".join(
    f"    WHEN '{k}' THEN {v}"
    for k, v in EVENT_WEIGHTS.items()
) + "\n    ELSE 0.0 END"

dedupe_where = ""
if DROP_EXACT_EVENT_DUPLICATES:
    dedupe_where = "WHERE rn = 1"

con.execute(f"""
CREATE OR REPLACE VIEW clean_events AS
WITH base AS (
    SELECT
        *,
        row_number() OVER (
            PARTITION BY
                user_id,
                item_id,
                event_id,
                ts,
                coalesce(session_id, ''),
                coalesce(sequence_id, '')
            ORDER BY source_file, source_row
        ) AS rn
    FROM standardized
    WHERE user_id IS NOT NULL
      AND item_id IS NOT NULL
      AND ts IS NOT NULL
      AND event_group <> 'other'
),
dedup AS (
    SELECT * EXCLUDE(rn)
    FROM base
    {dedupe_where}
)
SELECT
    *,
    {weight_case} AS event_weight,

    CASE
        WHEN event_group IN ('cart','offer','buy_start','buy_comp')
        THEN 1 ELSE 0
    END AS is_strong_positive,

    CASE
        WHEN event_group = 'buy_comp'
        THEN 1 ELSE 0
    END AS is_purchase,

    CASE
        WHEN ts < TIMESTAMP '{val_start:%Y-%m-%d %H:%M:%S}'
            THEN 'train'
        WHEN ts < TIMESTAMP '{test_start:%Y-%m-%d %H:%M:%S}'
            THEN 'val'
        ELSE 'test'
    END AS split
FROM dedup
""")

split_summary = q("""
SELECT
    split,
    count(*) AS interactions,
    count(DISTINCT user_id) AS users,
    count(DISTINCT item_id) AS items,
    count(*) FILTER (WHERE is_strong_positive=1) AS strong_events,
    count(*) FILTER (WHERE is_purchase=1) AS purchases,
    min(ts) AS min_ts,
    max(ts) AS max_ts
FROM clean_events
GROUP BY split
ORDER BY
    CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
""")

show("02_split_summary", split_summary)

02_split_summary: 3 rows


,split,interactions,users,items,strong_events,purchases,min_ts,max_ts
0,train,145998253,2581378,27328461,3948181,161381,2023-05-01,2023-05-25 23:59:59
1,val,16885121,989158,7275080,459485,18171,2023-05-26,2023-05-28 23:59:59
2,test,11842285,825830,5590766,306401,11755,2023-05-29,2023-05-31 00:00:00


,split,interactions,users,items,strong_events,purchases,min_ts,max_ts
0,train,145998253,2581378,27328461,3948181,161381,2023-05-01,2023-05-25 23:59:59
1,val,16885121,989158,7275080,459485,18171,2023-05-26,2023-05-28 23:59:59
2,test,11842285,825830,5590766,306401,11755,2023-05-29,2023-05-31 00:00:00


## 4. Xuất clean interactions theo train/validation/test

In [6]:
if INTERACTIONS_DIR.exists():
    shutil.rmtree(INTERACTIONS_DIR)

INTERACTIONS_DIR.mkdir(parents=True, exist_ok=True)

target = INTERACTIONS_DIR.as_posix().replace("'", "''")

con.execute(f"""
COPY (
    SELECT
        user_id,
        item_id,
        product_id,
        session_id,
        sequence_id,
        event_id,
        event_group,
        ts,
        price,
        name,
        category0,
        category1,
        category2,
        brand,
        condition,
        shipper,
        event_weight,
        is_strong_positive,
        is_purchase,
        split
    FROM clean_events
)
TO '{target}'
(
    FORMAT PARQUET,
    PARTITION_BY(split),
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 500000
)
""")

print("Saved:", INTERACTIONS_DIR)

Saved: D:\MerRec\data\processed\recommender\model_data\interactions


In [7]:
import polars as pl
from pathlib import Path

BASE = Path(
    r"D:\MerRec\data\processed\recommender\model_data"
)

for split in ["train", "val", "test"]:
    print(f"\n=== {split.upper()} ===")

    print(
        pl.scan_parquet(
            str(BASE / "interactions" / f"split={split}" / "*.parquet")
        )
        .group_by("event_group")
        .len()
        .sort("len", descending=True)
        .collect(engine="streaming")
    )


=== TRAIN ===
shape: (6, 2)
┌─────────────┬───────────┐
│ event_group ┆ len       │
│ ---         ┆ ---       │
│ str         ┆ u32       │
╞═════════════╪═══════════╡
│ view        ┆ 123706829 │
│ like        ┆ 18343243  │
│ cart        ┆ 2413351   │
│ offer       ┆ 1022470   │
│ buy_start   ┆ 350979    │
│ buy_comp    ┆ 161381    │
└─────────────┴───────────┘

=== VAL ===
shape: (6, 2)
┌─────────────┬──────────┐
│ event_group ┆ len      │
│ ---         ┆ ---      │
│ str         ┆ u32      │
╞═════════════╪══════════╡
│ view        ┆ 14200947 │
│ like        ┆ 2224689  │
│ cart        ┆ 279422   │
│ offer       ┆ 119819   │
│ buy_start   ┆ 42073    │
│ buy_comp    ┆ 18171    │
└─────────────┴──────────┘

=== TEST ===
shape: (6, 2)
┌─────────────┬─────────┐
│ event_group ┆ len     │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ view        ┆ 9948972 │
│ like        ┆ 1586912 │
│ cart        ┆ 186393  │
│ offer       ┆ 80627   │
│ buy_start   ┆ 27626 

## 5. Train-only ID maps và cold-start cohorts

In [8]:
train_view = f"""
read_parquet(
    '{(INTERACTIONS_DIR / "split=train" / "*.parquet").as_posix()}'
)
"""

val_view = f"""
read_parquet(
    '{(INTERACTIONS_DIR / "split=val" / "*.parquet").as_posix()}'
)
"""

test_view = f"""
read_parquet(
    '{(INTERACTIONS_DIR / "split=test" / "*.parquet").as_posix()}'
)
"""

# Xóa file map cũ nếu lần trước chạy lỗi hoặc bị 0 byte
for _p in [user_map_path, item_map_path]:
    if _p.exists():
        _p.unlink()

# ============================================================
# USER MAP
# ============================================================
con.execute(f"""
COPY (
    SELECT
        user_id,
        row_number() OVER (ORDER BY user_id) - 1 AS user_idx
    FROM (
        SELECT DISTINCT user_id
        FROM {train_view}
    )
)
TO '{user_map_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

# ============================================================
# ITEM MAP
# ============================================================
con.execute(f"""
COPY (
    SELECT
        item_id,
        row_number() OVER (ORDER BY item_id) - 1 AS item_idx
    FROM (
        SELECT DISTINCT item_id
        FROM {train_view}
    )
)
TO '{item_map_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

# ============================================================
# COLD-START SUMMARY
# ============================================================
cold_start = q(f"""
WITH train_users AS (
    SELECT DISTINCT user_id
    FROM {train_view}
),

train_items AS (
    SELECT DISTINCT item_id
    FROM {train_view}
),

eval AS (
    SELECT 'val' AS split, *
    FROM {val_view}

    UNION ALL

    SELECT 'test' AS split, *
    FROM {test_view}
)

SELECT
    split,
    count(*) AS interactions,

    100.0 * avg(
        CASE
            WHEN tu.user_id IS NULL THEN 1
            ELSE 0
        END
    ) AS pct_cold_user,

    100.0 * avg(
        CASE
            WHEN ti.item_id IS NULL THEN 1
            ELSE 0
        END
    ) AS pct_cold_item,

    100.0 * avg(
        CASE
            WHEN tu.user_id IS NULL
             AND ti.item_id IS NULL
            THEN 1
            ELSE 0
        END
    ) AS pct_both_cold

FROM eval e

LEFT JOIN train_users tu
    USING(user_id)

LEFT JOIN train_items ti
    USING(item_id)

GROUP BY split
ORDER BY split
""")

show(
    "03_cold_start_summary",
    cold_start
)

03_cold_start_summary: 2 rows


,split,interactions,pct_cold_user,pct_cold_item,pct_both_cold
0,test,11842285,6.353191,32.754152,1.636669
1,val,16885121,5.021717,24.710815,0.954776


,split,interactions,pct_cold_user,pct_cold_item,pct_both_cold
0,test,11842285,6.353191,32.754152,1.636669
1,val,16885121,5.021717,24.710815,0.954776


## 6. Item catalog cho Content-Based và giao diện

In [9]:
# ============================================================
# ITEM CATALOG - CHECKPOINT VERSION
#
# Mục tiêu:
# - phần nào làm xong -> lưu parquet ngay
# - chạy lại -> phần đã xong SKIP
# - không quay lại raw / clean_events
# ============================================================

from pathlib import Path


def valid_parquet(path):
    path = Path(path)
    return (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    )


# ============================================================
# TEMP CHECKPOINT PATHS
# ============================================================

catalog_val_path = SERVING_DIR / "_checkpoint_catalog_val.parquet"
catalog_test_path = SERVING_DIR / "_checkpoint_catalog_test.parquet"


# ============================================================
# HÀM TẠO CATALOG CHO 1 SPLIT
# ============================================================

def build_catalog(source_view, output_path, label):

    output_path = Path(output_path)

    if valid_parquet(output_path):
        print(f"✅ {label} đã tồn tại → SKIP")
        return

    # nếu lần trước lỗi để lại file 0 byte
    if output_path.exists():
        output_path.unlink()

    print(f"⏳ Đang tạo {label}...")

    con.execute(f"""
    COPY (
        SELECT
            item_id,

            arg_max(product_id, ts) AS product_id,
            arg_max(name, ts) AS name,
            arg_max(price, ts) AS price,

            arg_max(category0, ts) AS category0,
            arg_max(category1, ts) AS category1,
            arg_max(category2, ts) AS category2,

            arg_max(brand, ts) AS brand,
            arg_max(condition, ts) AS condition,
            arg_max(shipper, ts) AS shipper,

            max(ts) AS last_seen_ts

        FROM {source_view}

        GROUP BY item_id
    )
    TO '{output_path.as_posix()}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """)

    print(f"✅ {label} xong")


# ============================================================
# 1. TRAIN CATALOG
# ============================================================

build_catalog(
    train_view,
    catalog_train_path,
    "Train catalog"
)


# ============================================================
# 2. VAL CATALOG CHECKPOINT
# ============================================================

build_catalog(
    val_view,
    catalog_val_path,
    "Validation catalog"
)


# ============================================================
# 3. TEST CATALOG CHECKPOINT
# ============================================================

build_catalog(
    test_view,
    catalog_test_path,
    "Test catalog"
)


# ============================================================
# 4. GHÉP TRAIN + VAL + TEST
#
# Mỗi split đã chỉ còn 1 dòng / item.
# Bước này nhỏ hơn rất nhiều so với group 175M interaction.
# ============================================================

if valid_parquet(catalog_full_path):

    print("✅ Serving catalog đã tồn tại → SKIP")

else:

    if catalog_full_path.exists():
        catalog_full_path.unlink()

    print("⏳ Đang ghép Serving catalog...")

    con.execute(f"""
    COPY (
        SELECT
            item_id,

            arg_max(product_id, last_seen_ts) AS product_id,
            arg_max(name, last_seen_ts) AS name,
            arg_max(price, last_seen_ts) AS price,

            arg_max(category0, last_seen_ts) AS category0,
            arg_max(category1, last_seen_ts) AS category1,
            arg_max(category2, last_seen_ts) AS category2,

            arg_max(brand, last_seen_ts) AS brand,
            arg_max(condition, last_seen_ts) AS condition,
            arg_max(shipper, last_seen_ts) AS shipper,

            max(last_seen_ts) AS last_seen_ts

        FROM (
            SELECT *
            FROM read_parquet(
                '{catalog_train_path.as_posix()}'
            )

            UNION ALL

            SELECT *
            FROM read_parquet(
                '{catalog_val_path.as_posix()}'
            )

            UNION ALL

            SELECT *
            FROM read_parquet(
                '{catalog_test_path.as_posix()}'
            )
        )

        GROUP BY item_id
    )
    TO '{catalog_full_path.as_posix()}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """)

    print("✅ Serving catalog xong")


# ============================================================
# SUMMARY
# ============================================================

print()
print("Train catalog :", catalog_train_path)
print("Val checkpoint:", catalog_val_path)
print("Test checkpoint:", catalog_test_path)
print("Serving final :", catalog_full_path)

✅ Train catalog đã tồn tại → SKIP
✅ Validation catalog đã tồn tại → SKIP
✅ Test catalog đã tồn tại → SKIP
✅ Serving catalog đã tồn tại → SKIP

Train catalog : D:\MerRec\data\processed\recommender\model_data\item_catalog_train.parquet
Val checkpoint: D:\MerRec\data\processed\recommender\serving\_checkpoint_catalog_val.parquet
Test checkpoint: D:\MerRec\data\processed\recommender\serving\_checkpoint_catalog_test.parquet
Serving final : D:\MerRec\data\processed\recommender\serving\item_catalog_full.parquet


In [10]:
# ============================================================
# REBUILD ONLY VALIDATION CATALOG
# ============================================================

from pathlib import Path

catalog_val_path = (
    SERVING_DIR / "_checkpoint_catalog_val.parquet"
)

if catalog_val_path.exists():
    catalog_val_path.unlink()

print("⏳ Đang tạo lại Validation catalog...")

con.execute(f"""
COPY (
    SELECT
        item_id,

        arg_max(product_id, ts) AS product_id,
        arg_max(name, ts) AS name,
        arg_max(price, ts) AS price,

        arg_max(category0, ts) AS category0,
        arg_max(category1, ts) AS category1,
        arg_max(category2, ts) AS category2,

        arg_max(brand, ts) AS brand,
        arg_max(condition, ts) AS condition,
        arg_max(shipper, ts) AS shipper,

        max(ts) AS last_seen_ts

    FROM {val_view}

    GROUP BY item_id
)
TO '{catalog_val_path.as_posix()}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

print("✅ Validation catalog xong")
print("Path :", catalog_val_path)
print(
    "Size :",
    round(catalog_val_path.stat().st_size / 1024**2, 2),
    "MB"
)

⏳ Đang tạo lại Validation catalog...
✅ Validation catalog xong
Path : D:\MerRec\data\processed\recommender\serving\_checkpoint_catalog_val.parquet
Size : 276.48 MB


In [11]:
# ============================================================
# MERGE TRAIN + VAL + TEST -> SERVING CATALOG
# ============================================================

from pathlib import Path

train_p = Path(catalog_train_path).as_posix()
val_p   = Path(catalog_val_path).as_posix()
test_p  = Path(catalog_test_path).as_posix()
full_p  = Path(catalog_full_path).as_posix()

if Path(catalog_full_path).exists():
    Path(catalog_full_path).unlink()

print("⏳ Đang ghép Serving catalog...")

con.execute(f"""
COPY (
    SELECT
        item_id,

        arg_max(product_id, last_seen_ts) AS product_id,
        arg_max(name, last_seen_ts) AS name,
        arg_max(price, last_seen_ts) AS price,

        arg_max(category0, last_seen_ts) AS category0,
        arg_max(category1, last_seen_ts) AS category1,
        arg_max(category2, last_seen_ts) AS category2,

        arg_max(brand, last_seen_ts) AS brand,
        arg_max(condition, last_seen_ts) AS condition,
        arg_max(shipper, last_seen_ts) AS shipper,

        max(last_seen_ts) AS last_seen_ts

    FROM read_parquet([
        '{train_p}',
        '{val_p}',
        '{test_p}'
    ])

    GROUP BY item_id
)
TO '{full_p}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

print("✅ Serving catalog xong")
print("📦", catalog_full_path)

⏳ Đang ghép Serving catalog...
✅ Serving catalog xong
📦 D:\MerRec\data\processed\recommender\serving\item_catalog_full.parquet


## 7. Implicit-feedback train data cho ALS

In [13]:
con.execute(f"""
COPY (
    SELECT
        user_id,
        item_id,
        sum(event_weight) AS implicit_score,
        count(*) AS interactions,
        count(*) FILTER (WHERE event_group='view') AS views,
        count(*) FILTER (WHERE event_group='like') AS likes,
        count(*) FILTER (WHERE event_group='cart') AS carts,
        count(*) FILTER (WHERE event_group='offer') AS offers,
        count(*) FILTER (WHERE event_group='buy_start') AS buy_starts,
        count(*) FILTER (WHERE event_group='buy_comp') AS purchases,
        max(ts) AS last_ts
    FROM {train_view}
    GROUP BY user_id, item_id
)
TO '{cf_train_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

cf_summary = q(f"""
SELECT
    count(*) AS user_item_pairs,
    count(DISTINCT user_id) AS users,
    count(DISTINCT item_id) AS items,
    avg(implicit_score) AS mean_score,
    max(implicit_score) AS max_score
FROM read_parquet('{cf_train_path.as_posix()}')
""")

show("04_cf_train_summary", cf_summary)

04_cf_train_summary: 1 rows


,user_item_pairs,users,items,mean_score,max_score
0,109718519,2581378,27328461,1.633541,1092.0


,user_item_pairs,users,items,mean_score,max_score
0,109718519,2581378,27328461,1.633541,1092.0


## 8. Popularity baseline — chỉ dùng train cho offline evaluation

In [14]:
con.execute(f"""
COPY (
    SELECT
        item_id,
        sum(event_weight) AS popularity_score,
        count(*) AS interactions,
        count(DISTINCT user_id) AS unique_users,
        count(*) FILTER (WHERE event_group='view') AS views,
        count(*) FILTER (WHERE event_group='like') AS likes,
        count(*) FILTER (WHERE event_group='cart') AS carts,
        count(*) FILTER (WHERE event_group='offer') AS offers,
        count(*) FILTER (WHERE event_group='buy_start') AS buy_starts,
        count(*) FILTER (WHERE event_group='buy_comp') AS purchases,
        max(ts) AS last_event_ts
    FROM {train_view}
    GROUP BY item_id
    ORDER BY popularity_score DESC
)
TO '{pop_train_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

top_pop = q(f"""
SELECT *
FROM read_parquet('{pop_train_path.as_posix()}')
ORDER BY popularity_score DESC
LIMIT 50
""")

show("05_top_popularity_train", top_pop, n=20)

05_top_popularity_train: 50 rows


,item_id,popularity_score,interactions,unique_users,views,likes,carts,offers,buy_starts,purchases,last_event_ts
0,170090739,2519.0,1241,262,767,159,231,39,45,0,2023-05-25 22:37:14
1,31232418,1808.0,1644,1108,1502,132,8,2,0,0,2023-05-25 23:20:11
2,233858127,1698.0,1570,900,1467,92,10,0,1,0,2023-05-25 23:59:40
3,199825945,1624.0,1288,599,1079,168,0,39,2,0,2023-05-25 23:41:12
4,28325723,1309.0,1071,466,925,114,16,10,6,0,2023-05-25 00:53:04
5,157875024,1308.0,1288,1121,1270,17,1,0,0,0,2023-05-25 22:50:38
6,127650015,1292.0,1184,901,1094,84,0,6,0,0,2023-05-16 16:56:15
7,99159344,1280.0,679,286,358,193,120,0,8,0,2023-05-25 21:50:27
8,238852844,1274.0,1164,1089,1136,11,2,0,14,1,2023-05-25 12:39:07
9,223984650,1260.0,1189,930,1132,50,7,0,0,0,2023-05-20 19:21:43


,item_id,popularity_score,interactions,unique_users,views,likes,carts,offers,buy_starts,purchases,last_event_ts
0,170090739,2519.0,1241,262,767,159,231,39,45,0,2023-05-25 22:37:14
1,31232418,1808.0,1644,1108,1502,132,8,2,0,0,2023-05-25 23:20:11
2,233858127,1698.0,1570,900,1467,92,10,0,1,0,2023-05-25 23:59:40
3,199825945,1624.0,1288,599,1079,168,0,39,2,0,2023-05-25 23:41:12
4,28325723,1309.0,1071,466,925,114,16,10,6,0,2023-05-25 00:53:04
5,157875024,1308.0,1288,1121,1270,17,1,0,0,0,2023-05-25 22:50:38
6,127650015,1292.0,1184,901,1094,84,0,6,0,0,2023-05-16 16:56:15
7,99159344,1280.0,679,286,358,193,120,0,8,0,2023-05-25 21:50:27
8,238852844,1274.0,1164,1089,1136,11,2,0,14,1,2023-05-25 12:39:07
9,223984650,1260.0,1189,930,1132,50,7,0,0,0,2023-05-20 19:21:43


## 9. Trending — time-decay score

In [15]:
# ============================================================
# TRENDING - OPTIMIZED
# Không quay lại clean_events/raw
# Có checkpoint: file đã tồn tại thì SKIP
# ============================================================

from pathlib import Path
import pandas as pd


def valid_file(path):
    path = Path(path)
    return (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    )


# ============================================================
# Nguồn processed
# ============================================================

all_processed_view = f"""
read_parquet(
    '{(INTERACTIONS_DIR / "*" / "*.parquet").as_posix()}'
)
"""


# ============================================================
# Hàm build trending
# ============================================================

def build_trending(
    source_sql,
    reference_ts,
    output_path,
    label="Trending"
):
    output_path = Path(output_path)

    # Đã làm rồi thì bỏ qua
    if valid_file(output_path):
        print(f"✅ {label} đã tồn tại → SKIP")
        return

    # File lỗi / 0 byte từ lần trước
    if output_path.exists():
        output_path.unlink()

    ref = pd.Timestamp(reference_ts)
    half_life_seconds = TREND_HALF_LIFE_DAYS * 86400.0

    print(f"⏳ Đang tạo {label}...")

    con.execute(f"""
    COPY (
        SELECT
            item_id,

            SUM(
                event_weight * POW(
                    0.5,
                    date_diff(
                        'second',
                        ts,
                        TIMESTAMP '{ref:%Y-%m-%d %H:%M:%S}'
                    ) / {half_life_seconds}
                )
            ) AS trending_score,

            SUM(event_weight) AS weighted_score,

            COUNT(*) AS interactions,

            COUNT(DISTINCT user_id) AS unique_users,

            COUNT(*) FILTER (
                WHERE ts >=
                    TIMESTAMP '{ref:%Y-%m-%d %H:%M:%S}'
                    - INTERVAL '1 day'
            ) AS interactions_1d,

            COUNT(*) FILTER (
                WHERE ts >=
                    TIMESTAMP '{ref:%Y-%m-%d %H:%M:%S}'
                    - INTERVAL '3 days'
            ) AS interactions_3d,

            COUNT(*) FILTER (
                WHERE ts >=
                    TIMESTAMP '{ref:%Y-%m-%d %H:%M:%S}'
                    - INTERVAL '7 days'
            ) AS interactions_7d,

            COUNT(*) FILTER (
                WHERE event_group = 'buy_comp'
                  AND ts >=
                    TIMESTAMP '{ref:%Y-%m-%d %H:%M:%S}'
                    - INTERVAL '7 days'
            ) AS purchases_7d,

            MAX(ts) AS last_event_ts

        FROM {source_sql}

        GROUP BY item_id
    )
    TO '{output_path.as_posix()}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """)

    print(f"✅ {label} xong")


# ============================================================
# 1. OFFLINE TRENDING
# Chỉ TRAIN -> dùng cho evaluate
# ============================================================

train_max_ts = q(f"""
SELECT MAX(ts) AS x
FROM {train_view}
""").loc[0, "x"]

build_trending(
    train_view,
    train_max_ts,
    trend_train_path,
    "Train trending"
)


# ============================================================
# 2. SERVING TRENDING
# Train + Val + Test processed
# Không đọc clean_events
# ============================================================

serving_max_ts = q(f"""
SELECT MAX(ts) AS x
FROM {all_processed_view}
""").loc[0, "x"]

build_trending(
    all_processed_view,
    serving_max_ts,
    trend_serving_path,
    "Serving trending"
)


# ============================================================
# 3. TOP TRENDING
# Chỉ sort khi thực sự cần lấy Top 50
# ============================================================

top_trending = q(f"""
SELECT *
FROM read_parquet(
    '{trend_serving_path.as_posix()}'
)
ORDER BY trending_score DESC
LIMIT 50
""")

show(
    "06_top_trending_serving",
    top_trending,
    n=20
)

✅ Train trending đã tồn tại → SKIP
✅ Serving trending đã tồn tại → SKIP
06_top_trending_serving: 50 rows


,item_id,trending_score,weighted_score,interactions,unique_users,interactions_1d,interactions_3d,interactions_7d,purchases_7d,last_event_ts
0,170090739,360.086682,1945.0,1234,299,66,148,346,0,2023-05-30 23:39:54
1,118359303,304.365222,508.0,492,451,40,492,492,0,2023-05-30 22:39:40
2,122953576,300.474190,458.0,333,218,109,242,316,1,2023-05-30 22:58:18
3,194351263,296.280363,441.0,408,314,73,408,408,0,2023-05-30 23:52:34
4,152836185,296.009789,450.0,363,202,48,363,363,0,2023-05-30 23:34:19
5,110570801,283.350277,432.0,418,360,72,418,418,0,2023-05-30 23:52:33
6,201742207,266.660474,360.0,349,281,93,349,349,0,2023-05-30 23:48:12
7,63035183,262.920109,490.0,456,357,30,224,456,0,2023-05-30 23:51:20
8,140654332,259.974775,416.0,351,215,55,277,351,0,2023-05-30 23:10:14
9,40658225,259.886323,495.0,396,184,50,168,396,0,2023-05-30 23:22:48


,item_id,trending_score,weighted_score,interactions,unique_users,interactions_1d,interactions_3d,interactions_7d,purchases_7d,last_event_ts
0,170090739,360.086682,1945.0,1234,299,66,148,346,0,2023-05-30 23:39:54
1,118359303,304.365222,508.0,492,451,40,492,492,0,2023-05-30 22:39:40
2,122953576,300.474190,458.0,333,218,109,242,316,1,2023-05-30 22:58:18
3,194351263,296.280363,441.0,408,314,73,408,408,0,2023-05-30 23:52:34
4,152836185,296.009789,450.0,363,202,48,363,363,0,2023-05-30 23:34:19
5,110570801,283.350277,432.0,418,360,72,418,418,0,2023-05-30 23:52:33
6,201742207,266.660474,360.0,349,281,93,349,349,0,2023-05-30 23:48:12
7,63035183,262.920109,490.0,456,357,30,224,456,0,2023-05-30 23:51:20
8,140654332,259.974775,416.0,351,215,55,277,351,0,2023-05-30 23:10:14
9,40658225,259.886323,495.0,396,184,50,168,396,0,2023-05-30 23:22:48


## 10. Popularity/Trending snapshot dùng cho giao diện

In [ ]:
con.execute(f"""
COPY (
    SELECT
        item_id,
        sum(event_weight) AS popularity_score,
        count(*) AS interactions,
        count(DISTINCT user_id) AS unique_users,
        count(*) FILTER (WHERE event_group='buy_comp') AS purchases,
        max(ts) AS last_event_ts
    FROM clean_events
    GROUP BY item_id
    ORDER BY popularity_score DESC
)
TO '{pop_serving_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")


con.execute(f"""
COPY (
    WITH pop AS (
        SELECT
            item_id,
            popularity_score,
            row_number() OVER (
                ORDER BY popularity_score DESC
            ) AS popularity_rank
        FROM read_parquet('{pop_serving_path.as_posix()}')
    ),
    trend AS (
        SELECT
            item_id,
            trending_score,
            row_number() OVER (
                ORDER BY trending_score DESC
            ) AS trending_rank
        FROM read_parquet('{trend_serving_path.as_posix()}')
    ),
    catalog AS (
        SELECT *
        FROM read_parquet('{catalog_full_path.as_posix()}')
    )
    SELECT
        c.*,
        p.popularity_score,
        p.popularity_rank,
        t.trending_score,
        t.trending_rank
    FROM catalog c
    LEFT JOIN pop p USING(item_id)
    LEFT JOIN trend t USING(item_id)
    WHERE p.popularity_rank <= 1000
       OR t.trending_rank <= 1000
)
TO '{home_feed_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

home_preview = q(f"""
SELECT
    item_id,
    name,
    category2,
    brand,
    price,
    popularity_rank,
    trending_rank
FROM read_parquet('{home_feed_path.as_posix()}')
ORDER BY coalesce(trending_rank, 999999), coalesce(popularity_rank, 999999)
LIMIT 50
""")

show("07_home_feed_preview", home_preview, n=20)

07_home_feed_preview: 50 rows


,item_id,name,category2,brand,price,popularity_rank,trending_rank
0,170090739,Link's Awakening Amiibo,Action Figures & Accessories,Nintendo,19.0,2,1
1,118359303,Huge anime figure bundle 20+ figures,Action Figures,__UNK__,1199.0,505,2
2,122953576,PlayStation 5 Game Console with controllers,Consoles,Sony,360.0,804,3
3,194351263,Folklore Betty’s garden vinyl - Taylor Swift,Drums,Zildjian,125.0,930,4
4,152836185,YSL camera bag black leather shoulder bag,Crossbody Bags,YSL Yves Saint Laurent,890.0,868,5
5,110570801,Melanie Martinez Crybaby Perfume Milk - Read Desc,Women,Cry Baby,31.0,1015,6
6,201742207,3pc lps lot!,Play Animals,Littlest Pet Shop,19.0,2063,7
7,63035183,Wonderfold w4 luxe frame replacement,Other,Wonderworld,185.0,596,8
8,140654332,YSL Classic Monogram Crossbody Shoulder Bag in...,Shoulder Bags,YSL Yves Saint Laurent,397.0,1204,9
9,40658225,Ysl round leather shoulder bag crossbody bag i...,Shoulder Bags,YSL Yves Saint Laurent,769.0,563,10


,item_id,name,category2,brand,price,popularity_rank,trending_rank
0,170090739,Link's Awakening Amiibo,Action Figures & Accessories,Nintendo,19.0,2,1
1,118359303,Huge anime figure bundle 20+ figures,Action Figures,__UNK__,1199.0,505,2
2,122953576,PlayStation 5 Game Console with controllers,Consoles,Sony,360.0,804,3
3,194351263,Folklore Betty’s garden vinyl - Taylor Swift,Drums,Zildjian,125.0,930,4
4,152836185,YSL camera bag black leather shoulder bag,Crossbody Bags,YSL Yves Saint Laurent,890.0,868,5
5,110570801,Melanie Martinez Crybaby Perfume Milk - Read Desc,Women,Cry Baby,31.0,1015,6
6,201742207,3pc lps lot!,Play Animals,Littlest Pet Shop,19.0,2063,7
7,63035183,Wonderfold w4 luxe frame replacement,Other,Wonderworld,185.0,596,8
8,140654332,YSL Classic Monogram Crossbody Shoulder Bag in...,Shoulder Bags,YSL Yves Saint Laurent,397.0,1204,9
9,40658225,Ysl round leather shoulder bag crossbody bag i...,Shoulder Bags,YSL Yves Saint Laurent,769.0,563,10


## 11. Trending theo category

In [16]:
con.execute(f"""
COPY (
    WITH item_trend AS (
        SELECT
            t.item_id,
            t.trending_score,
            c.category0,
            c.category1,
            c.category2,
            c.name,
            c.brand,
            c.price
        FROM read_parquet('{trend_serving_path.as_posix()}') t
        JOIN read_parquet('{catalog_full_path.as_posix()}') c USING(item_id)
        WHERE c.category2 IS NOT NULL
    ),
    ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY category2
                ORDER BY trending_score DESC
            ) AS category_trending_rank
        FROM item_trend
    )
    SELECT *
    FROM ranked
    WHERE category_trending_rank <= 100
)
TO '{category_trending_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", category_trending_path)

Saved: D:\MerRec\data\processed\recommender\serving\trending_by_category.parquet


## 12. Cohort evaluation cho validation/test

In [17]:
con.execute(f"""
COPY (
    WITH train_users AS (
        SELECT DISTINCT user_id FROM {train_view}
    ),
    train_items AS (
        SELECT DISTINCT item_id FROM {train_view}
    ),
    eval AS (
        SELECT 'val' AS eval_split, * FROM {val_view}
        UNION ALL
        SELECT 'test' AS eval_split, * FROM {test_view}
    )
    SELECT
        e.*,

        CASE
            WHEN tu.user_id IS NULL AND ti.item_id IS NULL THEN 'cold_both'
            WHEN tu.user_id IS NULL THEN 'cold_user'
            WHEN ti.item_id IS NULL THEN 'cold_item'
            ELSE 'warm'
        END AS cohort

    FROM eval e
    LEFT JOIN train_users tu USING(user_id)
    LEFT JOIN train_items ti USING(item_id)
)
TO '{eval_cohort_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

cohort_summary = q(f"""
SELECT
    eval_split,
    cohort,
    count(*) AS interactions,
    count(DISTINCT user_id) AS users,
    count(DISTINCT item_id) AS items,
    count(*) FILTER (WHERE is_purchase=1) AS purchases
FROM read_parquet('{eval_cohort_path.as_posix()}')
GROUP BY 1,2
ORDER BY 1,2
""")

show("08_eval_cohort_summary", cohort_summary)

08_eval_cohort_summary: 8 rows


,eval_split,cohort,interactions,users,items,purchases
0,test,cold_both,193819,48063,129872,1564
1,test,cold_item,3685021,496690,1328698,2089
2,test,cold_user,558544,81064,397841,4214
3,test,warm,7404901,655210,4001445,3888
4,val,cold_both,161215,46849,112935,1582
5,val,cold_item,4011236,529301,1564451,2282
6,val,cold_user,686708,100818,470265,6767
7,val,warm,12025962,828148,5458198,7540


,eval_split,cohort,interactions,users,items,purchases
0,test,cold_both,193819,48063,129872,1564
1,test,cold_item,3685021,496690,1328698,2089
2,test,cold_user,558544,81064,397841,4214
3,test,warm,7404901,655210,4001445,3888
4,val,cold_both,161215,46849,112935,1582
5,val,cold_item,4011236,529301,1564451,2282
6,val,cold_user,686708,100818,470265,6767
7,val,warm,12025962,828148,5458198,7540


## 13. Kiểm tra leakage và integrity

In [18]:
integrity = q(f"""
WITH tr AS (
    SELECT min(ts) AS min_ts, max(ts) AS max_ts FROM {train_view}
),
va AS (
    SELECT min(ts) AS min_ts, max(ts) AS max_ts FROM {val_view}
),
te AS (
    SELECT min(ts) AS min_ts, max(ts) AS max_ts FROM {test_view}
)
SELECT
    tr.max_ts < va.min_ts AS train_before_val,
    va.max_ts < te.min_ts AS val_before_test,
    tr.min_ts AS train_min,
    tr.max_ts AS train_max,
    va.min_ts AS val_min,
    va.max_ts AS val_max,
    te.min_ts AS test_min,
    te.max_ts AS test_max
FROM tr, va, te
""")

show("09_split_integrity", integrity)

if not bool(integrity.loc[0, "train_before_val"]):
    raise ValueError("Leakage: train không kết thúc trước validation.")

if not bool(integrity.loc[0, "val_before_test"]):
    raise ValueError("Leakage: validation không kết thúc trước test.")

09_split_integrity: 1 rows


,train_before_val,val_before_test,train_min,train_max,val_min,val_max,test_min,test_max
0,True,True,2023-05-01,2023-05-25 23:59:59,2023-05-26,2023-05-28 23:59:59,2023-05-29,2023-05-31


## 14. Manifest output

In [19]:
manifest = pd.DataFrame([
    {
        "file": str(INTERACTIONS_DIR),
        "purpose": "Clean event-level data, partitioned train/val/test"
    },
    {
        "file": str(cf_train_path),
        "purpose": "ALS / LightFM implicit user-item matrix"
    },
    {
        "file": str(catalog_train_path),
        "purpose": "Train-only item metadata for Content-Based / Hybrid"
    },
    {
        "file": str(catalog_full_path),
        "purpose": "Serving item catalog"
    },
    {
        "file": str(pop_train_path),
        "purpose": "Offline Popularity baseline from train only"
    },
    {
        "file": str(trend_train_path),
        "purpose": "Offline Trending baseline from train only"
    },
    {
        "file": str(pop_serving_path),
        "purpose": "Popularity snapshot for website/demo"
    },
    {
        "file": str(trend_serving_path),
        "purpose": "Trending snapshot for website/demo"
    },
    {
        "file": str(home_feed_path),
        "purpose": "Top popularity/trending items joined with product metadata"
    },
    {
        "file": str(category_trending_path),
        "purpose": "Trending top items per category"
    },
    {
        "file": str(eval_cohort_path),
        "purpose": "Validatjion/test events labelled warm/cold-user/cold-item"
    },
    {
        "file": str(user_map_path),
        "purpose": "Train users -> contiguous integer index"
    },
    {
        "file": str(item_map_path),
        "purpose": "Train items -> contiguous integer index"
    },
])

show("10_output_manifest", manifest, n=50)
print("\n✅ PREPARATION COMPLETE")

10_output_manifest: 13 rows


,file,purpose
0,D:\MerRec\data\processed\recommender\model_dat...,"Clean event-level data, partitioned train/val/..."
1,D:\MerRec\data\processed\recommender\model_dat...,ALS / LightFM implicit user-item matrix
2,D:\MerRec\data\processed\recommender\model_dat...,Train-only item metadata for Content-Based / H...
3,D:\MerRec\data\processed\recommender\serving\i...,Serving item catalog
4,D:\MerRec\data\processed\recommender\model_dat...,Offline Popularity baseline from train only
5,D:\MerRec\data\processed\recommender\model_dat...,Offline Trending baseline from train only
6,D:\MerRec\data\processed\recommender\serving\p...,Popularity snapshot for website/demo
7,D:\MerRec\data\processed\recommender\serving\t...,Trending snapshot for website/demo
8,D:\MerRec\data\processed\recommender\serving\h...,Top popularity/trending items joined with prod...
9,D:\MerRec\data\processed\recommender\serving\t...,Trending top items per category



✅ PREPARATION COMPLETE


# Model strategy

So sánh theo đúng tầng của recommender:

**Baseline / fallback:** Popularity, Trending, Content-Based, Item-to-Item Co-visitation.

**Candidate retrieval:** ALS, LightFM Hybrid, Two-Tower Hybrid, SASRec.

**Final ranking:** LightGBM Ranker là lựa chọn chính; DeepFM và DCN dùng làm benchmark bổ sung nếu còn thời gian/compute.

Đánh giá candidate models bằng Recall@K, HitRate@K, NDCG@K và tách riêng warm / cold-user / cold-item. Đánh giá ranker trên cùng candidate pool bằng NDCG@K, MRR và Recall@K.